# NOAA GEFS — Download

**Model:** NOAA Global Ensemble Forecast System  
**Type:** Physics-based probabilistic — 31 members (1 control + 30 perturbed)  
**Cycles:** 00Z and 12Z only  
**Library:** `herbie-data`

## Files produced

```
data/YYYY-MM-DD_HHZ/
    gefs_mean_f000-f240_6h.grib2     ensemble mean (6-hourly, 0–240h)
    gefs_spread_f000-f240_6h.grib2   ensemble spread
    gefs_prob_precip_d1-d5.nc        P(precip>1mm) and P(>5mm), D1–D5 windows
```

In [ ]:
from herbie import Herbie
from pathlib import Path
import pandas as pd
import xarray as xr

BASE_DIR = Path("data")
BASE_DIR.mkdir(parents=True, exist_ok=True)
print("Ready. BASE_DIR:", BASE_DIR.resolve())

In [ ]:
# GEFS only runs at 00Z and 12Z
def find_latest_gefs(max_lookback_h: int = 36) -> pd.Timestamp:
    now = pd.Timestamp.now("UTC").replace(tzinfo=None)
    for lag_h in range(0, max_lookback_h + 1):
        candidate = (now - pd.Timedelta(hours=lag_h)).floor("12h")
        try:
            H = Herbie(candidate, model="gefs", member="mean",
                       product="atmos.25", fxx=0, verbose=False)
            inv = H.inventory()
            if inv is not None and len(inv) > 0:
                return candidate
        except Exception:
            continue
    raise RuntimeError(f"No GEFS run found in the last {max_lookback_h} h")

latest_run  = find_latest_gefs()
latest_time = latest_run.hour

run_label  = f"{latest_run.strftime('%Y-%m-%d')}_{latest_time:02d}Z"
output_dir = BASE_DIR / run_label
output_dir.mkdir(parents=True, exist_ok=True)
tmp_dir    = output_dir / ".tmp"
tmp_dir.mkdir(exist_ok=True)

print("Latest run  :", latest_run)
print("Output dir  :", output_dir.resolve())

In [ ]:
# ── GEFS mean + spread — 6-hourly, 0–120h ────────────────────────────────────
steps_6h = list(range(0, 121, 6))

GEFS = {}
GEFS["thermo"]      = (":TMP:2 m above ground:|"
                       ":DPT:2 m above ground:|"
                       ":RH:2 m above ground:")
GEFS["wind"]        = (":UGRD:10 m above ground:|"
                       ":VGRD:10 m above ground:|"
                       ":GUST:surface:")
GEFS["pressure"]    = (":PRMSL:mean sea level:|"
                       ":PRES:surface:")
GEFS["moisture"]    = (":APCP:surface:|"
                       ":PWAT:entire atmosphere (considered as a single layer):|"
                       ":TCDC:entire atmosphere (considered as a single layer):")
GEFS["energy"]      = (":ULWRF:top of atmosphere:|"
                       ":SHTFL:surface:|"
                       ":LHTFL:surface:|"
                       ":DSWRF:surface:|"
                       ":DLWRF:surface:")
GEFS["instability"] = (":CAPE:surface:|"
                       ":CIN:surface:")

GEFS_SEARCH = "|".join(GEFS.values())
print(f"Downloading {len(steps_6h)} steps — {len(GEFS)} groups")

for member, label in [("mean", "gefs_mean"), ("spread", "gefs_spread")]:
    parts = []
    for fxx in steps_6h:
        H = Herbie(latest_run, model="gefs", member=member,
                   product="atmos.25", fxx=fxx, verbose=False)
        p = Path(H.download(GEFS_SEARCH, save_dir=tmp_dir))
        parts.append(p)
        print(f"  f{fxx:03d} ✓", end="  ")
    print()
    target = output_dir / f"{label}_f000-f120_6h.grib2"
    with open(target, "wb") as out:
        for p in parts: out.write(p.read_bytes())
    for p in parts: p.unlink(missing_ok=True)
    print(f"Saved: {target.name}  ({target.stat().st_size // 1024} KB)")

In [ ]:
# ── GEFS precipitation probability — empirical from 31 members ───────────────
TGPY_LAT, TGPY_LON = 12.0042, -61.7862
members      = ["c00"] + [f"p{i:02d}" for i in range(1, 31)]
windows      = [(s, s + 24) for s in range(0, 121, 24)]  # D1–D5
needed_steps = sorted({fxx for s, e in windows for fxx in (s, e)})

apcp_cache = {m: {} for m in members}
print(f"Downloading APCP for 31 members × {len(needed_steps)} steps...")

for m in members:
    for fxx in needed_steps:
        H  = Herbie(latest_run, model="gefs", member=m,
                    product="atmos.25", fxx=fxx, verbose=False)
        ds = H.xarray(":APCP:surface:")
        pt = ds.sel(latitude=TGPY_LAT, longitude=TGPY_LON,
                    method="nearest", tolerance=0.5)
        var = [v for v in pt.data_vars if "tp" in v or "apcp" in v.lower()][0]
        apcp_cache[m][fxx] = float(pt[var].values)
    print(f"  {m} ✓", end="  ")
print()

thresholds = {"tpg1": 1.0, "tpg5": 5.0}
results    = {}
for label, thresh in thresholds.items():
    probs = []
    for s, e in windows:
        precip = [max(0.0, apcp_cache[m][e] - apcp_cache[m][s]) for m in members]
        probs.append(round(sum(p >= thresh for p in precip) / len(members), 4))
    results[label] = probs

window_labels = [f"{s}-{e}h" for s, e in windows]
ds_prob = xr.Dataset(
    {k: xr.DataArray(v, dims=["window"], coords={"window": window_labels})
     for k, v in results.items()},
    attrs={"description": "GEFS empirical precip probability at TGPY",
           "n_members": len(members), "run": run_label}
)
prob_path = output_dir / "gefs_prob_precip_d1-d5.nc"
ds_prob.to_netcdf(prob_path)
print(f"Saved: {prob_path.name}")
print(ds_prob.to_dataframe().round(3))